In [ ]:
#%%writefile hr_full_app.py
# ---------------------------------------------------------
# Komplexný HR Analytický Nástroj
# 1. Load Data (SQL)
# 2. Smart Model Loading/Training (XGBoost + Optuna)
# 3. Anomaly Detection (Regression)
# 4. Gradio Frontend (Simulator + Searchable Dashboard)
# ---------------------------------------------------------
import os, joblib, warnings, optuna
warnings.filterwarnings('ignore')
import urllib.parse
from pathlib import Path
DATA_ENV_PATH = Path("data.env")

import pandas as pd
import numpy as np
import sqlalchemy as sa

# Vizualizácie
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Machine Learning
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score, mean_absolute_error
from xgboost import XGBClassifier, XGBRegressor

# Frontend
import gradio as gr

def get_xgboost_device():
    try:
        X_test = np.array([[1, 2], [3, 4]])
        y_test = np.array([0, 1])
        test_model = XGBClassifier(device="cuda", n_estimators=1, tree_method="hist")
        test_model.fit(X_test, y_test)
        print("[SYSTEM] GPU (CUDA) detekované! Výpočty pobežia na grafickej karte.")
        return "cuda"
    except Exception:
        print("[SYSTEM] GPU (CUDA) nedostupné. Prepínam na CPU.")
        return "cpu"

ACTIVE_DEVICE = get_xgboost_device()

MODEL_PATH = "model_attrition.pkl"
ENCODER_PATH = "encoder_attrition.pkl"
FEATURE_COLS_PATH = "feature_cols.pkl"


# 1. SQL Engine & Data Load

def load_env_from_file(env_path: Path = DATA_ENV_PATH) -> None:
    if not env_path.exists():
        raise FileNotFoundError(f"Env file not found: {env_path}")
    with env_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            os.environ[key.strip()] = value.strip()

def make_sql_engine() -> sa.Engine:
    load_env_from_file()
    required = ["SQL_SERVER", "SQL_DATABASE", "SQL_USERNAME", "SQL_PASSWORD", "SQL_DRIVER"]
    missing = [k for k in required if not os.getenv(k)]
    if missing:
        raise RuntimeError(f"Chýbajú premenné prostredia: {missing}")

    params = urllib.parse.quote_plus(
        f"DRIVER={{{os.getenv('SQL_DRIVER')}}};"
        f"SERVER={os.getenv('SQL_SERVER')};"
        f"DATABASE={os.getenv('SQL_DATABASE')};"
        f"UID={os.getenv('SQL_USERNAME')};"
        f"PWD={os.getenv('SQL_PASSWORD')};"
        f"Encrypt={os.getenv('SQL_ENCRYPT', 'no')};"
        f"TrustServerCertificate={os.getenv('SQL_TRUSTCERT', 'yes')};"
    )
    return sa.create_engine(f"mssql+pyodbc:///?odbc_connect={params}", fast_executemany=True)

def load_hr_dataframe() -> pd.DataFrame:
    engine = make_sql_engine()
    print("--> Načítavam dáta z SQL...")
    query = "SELECT * FROM dbo.HR_Synth_Data"
    df = pd.read_sql(query, engine)
    print(f"    Načítaných {len(df)} riadkov.")
    return df

def save_to_sql(df: pd.DataFrame, table_name: str):
    engine = make_sql_engine()
    print(f"--> Zapisujem tabuľku: {table_name} ({len(df)} riadkov)...")
    try:
        df.to_sql(table_name, engine, if_exists='replace', index=False)
        print("    Zápis úspešný.")
    except Exception as e:
        print(f"    Chyba pri zápise do SQL: {e}")

#2. Preprocessing & Helpers

def get_drop_columns():
    return ["EmployeeNumber", "EmployeeCount", "StandardHours", "Over18", 
            "FirstName", "LastName", "FullName", "Email", "Username"]

def preprocess_data_for_training(df: pd.DataFrame):
    df_proc = df.copy()
    drop_cols = get_drop_columns()
    
    # Target Encoding
    y = None
    if "Attrition" in df_proc.columns:
        df_proc["Attrition"] = df_proc["Attrition"].apply(lambda x: 1 if str(x).lower() == 'yes' else 0)
        y = df_proc["Attrition"]
        df_proc = df_proc.drop(columns=["Attrition"])
    
    df_proc = df_proc.drop(columns=[c for c in drop_cols if c in df_proc.columns], errors='ignore')

    encoders = {}
    cat_cols = df_proc.select_dtypes(include=["object", "category"]).columns

    for col in cat_cols:
        le = LabelEncoder()
        df_proc[col] = le.fit_transform(df_proc[col].astype(str))
        encoders[col] = le
        
    return df_proc, y, encoders

def preprocess_data_for_inference(df: pd.DataFrame, encoders, feature_cols):
    df_proc = df.copy()
    drop_cols = get_drop_columns() + ["Attrition"] # Attrition pri inference nepotrebujeme v X
    
    X = df_proc.drop(columns=[c for c in drop_cols if c in df_proc.columns], errors='ignore')
    
    for col, le in encoders.items():
        if col in X.columns:
            X[col] = X[col].astype(str).map(lambda s: le.transform([s])[0] if s in le.classes_ else 0)
            X[col] = X[col].fillna(0).astype(int)
            
    for col in feature_cols:
        if col not in X.columns:
            X[col] = 0
            
    return X[feature_cols]

def run_optuna_optimization(X_train, y_train, X_test, y_test, n_trials=100):
    def objective(trial: optuna.Trial) -> float:
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 1200),
            "max_depth": trial.suggest_int("max_depth", 3, 15),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "gamma": trial.suggest_float("gamma", 0, 5),
            "random_state": 42,
            "n_jobs": -1,
            "tree_method": "hist",
            "objective": "binary:logistic",
            "eval_metric": "logloss", # Pre klasifikáciu
            "early_stopping_rounds": 20,
            "device": ACTIVE_DEVICE,
        }
        model = XGBClassifier(**params)
        model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
        
        preds = model.predict(X_test)
        return accuracy_score(y_test, preds)

    print(f"--> Spúšťam Optuna optimalizáciu ({n_trials} pokusov)...")
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    
    print(f"    Najlepšia Accuracy: {study.best_value:.4f}")
    return study.best_params

def train_and_save_new_model(df_full: pd.DataFrame, n_trials=100):
    print("\n--- Spúšťam tréning nového modelu ---")
    
    # 1. Preprocessing
    X, y, encoders = preprocess_data_for_training(df_full)
    feature_names = X.columns.tolist()
    
    # 2. Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    # 3. Optuna
    best_params = run_optuna_optimization(X_train, y_train, X_test, y_test, n_trials)
    
    # 4. Final Train
    final_params = best_params.copy()
    final_params.update({"random_state": 42, "n_jobs": -1, "tree_method": "hist","objective": "binary:logistic", "eval_metric": "logloss", "early_stopping_rounds": 20, "device": ACTIVE_DEVICE,})
    
    print("--> Trénujem finálny model na celom datasete...")
    final_model = XGBClassifier(**final_params)
    final_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    
    acc = accuracy_score(y_test, final_model.predict(X_test))
    print(f"    Finálna Presnosť (Test set): {acc:.2%}")
    
    # 5. Uloženie
    print(f"--> Ukladám artefakty modelu do {MODEL_PATH}...")
    joblib.dump(final_model, MODEL_PATH)
    joblib.dump(encoders, ENCODER_PATH)
    joblib.dump(feature_names, FEATURE_COLS_PATH)
    
    return final_model, encoders, feature_names, acc

# 4. Smart Loading Logic

def get_model_and_predictions(df_full: pd.DataFrame):
    if os.path.exists(MODEL_PATH) and os.path.exists(ENCODER_PATH) and os.path.exists(FEATURE_COLS_PATH):
        print(f"\n[INFO] Nájdený uložený model ({MODEL_PATH}). Načítavam...")
        try:
            model = joblib.load(MODEL_PATH)
            encoders = joblib.load(ENCODER_PATH)
            feature_names = joblib.load(FEATURE_COLS_PATH)
            acc = 0.0 
        except Exception as e:
            print(f"[ERROR] Chyba pri načítaní: {e}. Spustím nový tréning.")
            model, encoders, feature_names, acc = train_and_save_new_model(df_full)
    else:
        print(f"\n[INFO] Model nenájdený. Spúšťam tréning...")
        model, encoders, feature_names, acc = train_and_save_new_model(df_full)

    print("--> Generujem predikcie pre aktuálne dáta...")
    X_current = preprocess_data_for_inference(df_full, encoders, feature_names)
    
    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": model.feature_importances_
    }).sort_values(by="Importance", ascending=False)
    
    predictions_df = df_full.copy()
    all_probs = model.predict_proba(X_current)[:, 1]
    
    predictions_df["Risk_Label"] = pd.cut(
        all_probs, 
        bins=[-0.1, 0.33, 0.50, 1.1], 
        labels=["Low", "Medium", "High"]
    )
    
    predictions_df["Attrition_Prob"] = (all_probs * 100).round(2)
    predictions_df["Attrition_Prob"] = predictions_df["Attrition_Prob"].astype(str) + "%"

    cols_to_drop = ["EmployeeCount", "StandardHours", "Over18", "FirstName", "LastName", "Email", "Username"]
    predictions_df.drop(columns=[c for c in cols_to_drop if c in predictions_df.columns], inplace=True)
    cols_order = ["EmployeeNumber", "FullName", "Risk_Label", "Attrition_Prob", "Department", "JobRole"]
    remaining_cols = [c for c in predictions_df.columns if c not in cols_order]
    predictions_df = predictions_df[cols_order + remaining_cols]
    
    return model, encoders, feature_names, acc, importance_df, predictions_df

def detect_department_anomalies(df_full: pd.DataFrame, n_optuna_trials=100):
    print("\n--- 2. Detekcia Anomálií (Regression + Optuna) ---")
    df_agg = df_full.copy()
    df_agg["Attrition_Flag"] = df_agg["Attrition"].apply(lambda x: 1 if str(x).lower() == 'yes' else 0)
    df_agg["OverTime_Flag"] = df_agg["OverTime"].apply(lambda x: 1 if str(x).lower() == 'yes' else 0)

    group_cols = ["Department", "JobRole"]
    
    dept_stats = df_agg.groupby(group_cols).agg({
        "EmployeeNumber": "count",
        "Attrition_Flag": "mean",      
        "OverTime_Flag": "mean", 
        "EnvironmentSatisfaction": lambda x: x.astype(int).mean(),
        "DailyRate": "mean",
        "WorkLifeBalance": lambda x: x.astype(int).mean()
    }).reset_index()

    dept_stats.rename(columns={
        "EmployeeNumber": "Headcount", 
        "Attrition_Flag": "Actual_Attrition_Rate",
        "OverTime_Flag": "Avg_OverTime"
    }, inplace=True)

    dept_stats = dept_stats[dept_stats["Headcount"] > 2].copy()

    features = ["Avg_OverTime", "EnvironmentSatisfaction", "DailyRate", "WorkLifeBalance"]
    X = dept_stats[features]
    y = dept_stats["Actual_Attrition_Rate"]

    if len(dept_stats) < 10:
        print("   [INFO] Málo dát pre Optunu (<10 skupín). Používam default XGBoost.")
        best_params = {"n_estimators": 100, "max_depth": 3, "learning_rate": 0.01,"objective": "reg:squarederror","device": ACTIVE_DEVICE}
    else:
        def objective_reg(trial):
            params = {
                "n_estimators": trial.suggest_int("n_estimators", 50, 1200),
                "max_depth": trial.suggest_int("max_depth", 3, 15),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
                "min_child_weight": trial.suggest_int("min_child_weight", 1, 5),
                "random_state": 42,
                "n_jobs": -1,
                "tree_method": "hist",
                "device": ACTIVE_DEVICE,
                "objective": "reg:squarederror" # Regresia
            }

            X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
            
            model = XGBRegressor(**params)
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
            
            preds = model.predict(X_val)
            rmse = np.sqrt(mean_squared_error(y_val, preds))
            return rmse

        print(f"--> Spúšťam Optunu pre Regresiu ({n_optuna_trials} pokusov)...")
        optuna.logging.set_verbosity(optuna.logging.WARNING)
        study_reg = optuna.create_study(direction="minimize")
        study_reg.optimize(objective_reg, n_trials=n_optuna_trials, show_progress_bar=True)
        best_params = study_reg.best_params
        print(f"    Najlepšie parametre (Regresia): {best_params}")

    final_params = best_params.copy()
    final_params.update({"random_state": 42, "n_jobs": -1, "objective": "reg:squarederror","device": ACTIVE_DEVICE})
    
    reg_model = XGBRegressor(**final_params)
    reg_model.fit(X, y)

    dept_stats["Predicted_Attrition_Rate"] = reg_model.predict(X)
    
    r2 = r2_score(y, dept_stats["Predicted_Attrition_Rate"])
    mae = mean_absolute_error(y, dept_stats["Predicted_Attrition_Rate"])
    
    print(f"R² Score modelu (Vysvetlená variabilita): {r2:.2%}")
    print(f"Priemerná chyba modelu (MAE): {mae:.2%}")

    dept_stats["Anomaly_Score"] = dept_stats["Actual_Attrition_Rate"] - dept_stats["Predicted_Attrition_Rate"]
    dept_stats["Is_Anomaly"] = dept_stats["Anomaly_Score"] > 0.10
    dept_stats = dept_stats.sort_values(by="Anomaly_Score", ascending=False)

    return dept_stats

# 6. Gradio Frontend
def run_gradio_app(model, encoders, feature_names, global_acc, importance_df, df_sample, full_predictions_df):
    
    def predict_new_employee(age, dept, role, overtime, income, env_sat, years_at_company):
        input_data = df_sample.iloc[0:1].copy()
        input_data["Age"] = age
        input_data["Department"] = dept
        input_data["JobRole"] = role
        input_data["OverTime"] = overtime
        input_data["MonthlyIncome"] = income
        input_data["EnvironmentSatisfaction"] = env_sat
        input_data["YearsAtCompany"] = years_at_company
        
        X_new = preprocess_data_for_inference(input_data, encoders, feature_names)

        prob = model.predict_proba(X_new)[0, 1]
        
        if prob <= 0.33:
            decision = "NÍZKE RIZIKO"
            color = "green"
        elif prob <= 0.50:
            decision = "MIERNE RIZIKO"
            color = "orange"
        else:
            decision = "VYSOKÉ RIZIKO"
            color = "red"
        result_md = f"""
        ### Výsledok Analýzy
        **Pravdepodobnosť odchodu:** {prob:.2%}
        **Verdikt:** <span style='color:{color}; font-weight:bold; font-size:16px'>{decision}</span>
        
        *(Celková presnosť modelu: ~{global_acc:.2%})*
        """
        
        fig = px.bar(importance_df.head(50), x='Importance', y='Feature', orientation='h', 
                        title="Top faktory (Feature Importance)", color='Importance')
        fig.update_layout(yaxis={'categoryorder':'total ascending'})
        return result_md, fig

    def filter_database(name_query, dept_query, id_query):
        df_filtered = full_predictions_df.copy()
        if name_query:
            df_filtered = df_filtered[df_filtered["FullName"].str.contains(name_query, case=False, na=False)]
        if dept_query and dept_query != "Všetky":
            df_filtered = df_filtered[df_filtered["Department"] == dept_query]
        if id_query:
            df_filtered = df_filtered[df_filtered["EmployeeNumber"].astype(str).str.contains(id_query, na=False)]
            
        count_msg = f" Nájdených **{len(df_filtered)}** z celkových {len(full_predictions_df)} zamestnancov."
        return df_filtered, count_msg

    with gr.Blocks(theme=gr.themes.Soft(), title="HR Analytics AI") as demo:
        gr.Markdown("# HR AI Analytics Platform")
        
        with gr.Tabs():
            with gr.Tab("Simulátor Nového Zamestnanca"):
                with gr.Row():
                    with gr.Column():
                        gr.Markdown("### Vstupné parametre")
                        in_age = gr.Slider(18, 65, value=30, label="Vek")
                        dept_choices = sorted(list(df_sample["Department"].unique()))
                        in_dept = gr.Dropdown(choices=dept_choices, label="Oddelenie", value=dept_choices[0])
                        in_role = gr.Dropdown(choices=sorted(list(df_sample["JobRole"].unique())), label="Pozícia")
                        in_over = gr.Radio(["Yes", "No"], label="Nadčasy", value="No")
                        in_income = gr.Slider(1000, 20000, value=5000, label="Príjem")
                        in_env = gr.Slider(1, 4, value=3, label="Spokojnosť (1-4)")
                        in_years = gr.Slider(0, 40, value=5, label="Roky vo firme")
                        
                        btn = gr.Button("Analyzovať Riziko", variant="primary")
                    
                    with gr.Column():
                        gr.Markdown("### Výstup modelu")
                        out_txt = gr.Markdown()
                        out_plt = gr.Plot()
                
                btn.click(predict_new_employee, 
                            inputs=[in_age, in_dept, in_role, in_over, in_income, in_env, in_years],
                            outputs=[out_txt, out_plt])

            with gr.Tab("Data Dashboard"):
                gr.Markdown("### Vyhľadávanie v predikciách")
                
                with gr.Row():
                    search_name = gr.Textbox(label="Meno zamestnanca", placeholder="Napr. John")
                    all_depts = ["Všetky"] + sorted(list(full_predictions_df["Department"].unique()))
                    search_dept = gr.Dropdown(choices=all_depts, label="Oddelenie", value="Všetky")
                    search_id = gr.Textbox(label="ID Zamestnanca", placeholder="Napr. 104")
                
                lbl_count = gr.Markdown(value=f"Nájdených **{len(full_predictions_df)}** z celkových {len(full_predictions_df)} záznamov.")
                
                data_table = gr.Dataframe(value=full_predictions_df, interactive=False)

                inputs = [search_name, search_dept, search_id]
                outputs = [data_table, lbl_count]
                
                search_name.change(filter_database, inputs, outputs)
                search_dept.change(filter_database, inputs, outputs)
                search_id.change(filter_database, inputs, outputs)

                gr.Markdown("---")
                gr.Markdown("### Anomálie (Podľa oddelení a pozícií)")
                try:
                    gr.Dataframe(value=anomalies_df)
                except:
                    gr.Markdown("*Tabuľka anomálií nie je dostupná v tomto kontexte.*")
    return demo

# MAIN Function
if __name__ == "__main__":
    print("HR Analytics Pipeline Started")
    
    try:
        df = load_hr_dataframe()
    except Exception as e:
        print(f"CRITICAL ERROR (DB): {e}")
        exit(1)

    model, encoders, feature_names, acc, feat_imp, predictions_df = get_model_and_predictions(df)

    anomalies_df = detect_department_anomalies(df)

    save_to_sql(predictions_df, "Predictions_Attrition")
    save_to_sql(feat_imp, "Model_Feature_Importance")
    save_to_sql(anomalies_df, "Anomalies_Department")

    print("\nVšetko hotovo. Spúšťam Gradio")
    app = run_gradio_app(model, encoders, feature_names, acc, feat_imp, df, predictions_df)
    app.launch(share=False)

[SYSTEM] GPU (CUDA) detekované! Výpočty pobežia na grafickej karte.
HR Analytics Pipeline Started
--> Načítavam dáta z SQL...
    Načítaných 8820 riadkov.

[INFO] Nájdený uložený model (model_attrition.pkl). Načítavam...
--> Generujem predikcie pre aktuálne dáta...


UFuncTypeError: ufunc 'add' did not contain a loop with signature matching types (dtype('<U32'), dtype('<U1')) -> None